# PSX Squeeze Ensemble — Strategy Walkthrough

Layer-by-layer signal generation on live PSO.KA data.

**Layers covered:**
1. Layer 0 — TTM Squeeze base signal
2. Layer 1 — SMA-200 trend filter + squeeze duration
3. Layer 2 — ADX momentum filter
4. Layer 3 — XGBoost TP-hit probability
5. Layer 4 — HMM regime filter

In [ ]:
import sys
sys.path.insert(0, '../src')

from data.fetcher import fetch_data
from squeeze.indicators import compute_indicators
from squeeze.filters import apply_rule_filters
from squeeze.signals import generate_base_signal, generate_rule_filtered_signal, generate_ensemble_signal
from ml.hmm_regime import fit_hmm, hmm_bull_states
from ml.xgboost_scorer import score_signals
from backtest.engine import backtest

SYMBOL = 'PSO'

In [ ]:
# Fetch data
df_raw = fetch_data(SYMBOL)
print(f'Fetched {len(df_raw)} bars for {SYMBOL}')
df_raw.tail()

In [ ]:
# Compute indicators
df = compute_indicators(df_raw)
df = apply_rule_filters(df)
df = generate_base_signal(df)
print(f'Base signals: {df["base_signal"].sum()}')

In [ ]:
# Apply ML layers
df['xgb_ok']   = score_signals(df, df['base_signal'])
hmm_model      = fit_hmm(df['log_ret'].values)
df['hmm_bull'] = hmm_bull_states(hmm_model, df['log_ret'].values)
df = generate_rule_filtered_signal(df)
df = generate_ensemble_signal(df)

print(f'Rule-filtered signals : {df["sig_rules"].sum()}')
print(f'Ensemble signals      : {df["sig_ensemble"].sum()}')

In [ ]:
# Compare results
for sig in ['base_signal', 'sig_rules', 'sig_ensemble']:
    r = backtest(df, sig, SYMBOL)
    print(f'{sig:20s}  Trades:{r["total_trades"]:3d}  '
          f'WR:{r["win_rate"]:5.1f}%  CAGR:{r["cagr_pct"]:6.1f}%  '
          f'MaxDD:{r["max_drawdown_pct"]:6.1f}%')